# VAE for Anomaly Detection - Factory Sensor Experiment

This notebook demonstrates using Variational Autoencoders (VAEs) for anomaly detection in industrial sensor data.

## Concept Overview

The VAE learns a compressed representation of "normal" sensor patterns. When new data arrives:
- **Normal data** is reconstructed accurately (low error)
- **Anomalous data** fails to reconstruct properly (high error)

We'll cover:
1. Data preparation (synthetic bearing vibration data)
2. Training the VAE on healthy data only
3. Anomaly detection using reconstruction error
4. Root cause analysis (which sensors contributed most to the anomaly)
5. Latent space visualization

In [ ]:
# Install dependencies if needed
# !pip install -r requirements.txt

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Add src to path
import sys
sys.path.insert(0, '.')

from src.data_loader import BearingDataLoader
from src.vae_model import VAE
from src.trainer import VAETrainer
from src.anomaly_detector import AnomalyDetector
from src.visualization import (
    LatentSpaceVisualizer,
    plot_reconstruction_error,
    plot_feature_contributions,
    plot_training_history,
    plot_anomaly_timeline
)

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'}")

## Step 1: Data Preparation

We'll generate synthetic bearing vibration data that simulates:
- 4 bearing sensors recording vibration
- Gradual degradation leading to failure in one bearing
- The model should detect this degradation using only "healthy" training data

In [ ]:
# Initialize data loader
loader = BearingDataLoader(
    data_dir="data",
    window_size=2048,    # Samples per window
    stride=1024,         # 50% overlap
    use_frequency_features=False  # Use raw time-domain features
)

# Load and prepare data
# Training set: ONLY healthy data (unsupervised learning)
# Test set: Mix of healthy + faulty data
X_train, X_test, y_train, y_test = loader.load_data(use_synthetic=True)

print(f"\nData shapes:")
print(f"  X_train: {X_train.shape} (all healthy)")
print(f"  X_test:  {X_test.shape}")
print(f"  Test labels: {int(sum(y_test == 0))} normal, {int(sum(y_test == 1))} anomaly")

In [ ]:
# Visualize a sample window
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Reshape to see 4 channels
n_channels = 4
window_size = X_train.shape[1] // n_channels

sample_healthy = X_train[0].reshape(window_size, n_channels)
sample_anomaly = X_test[y_test == 1][0].reshape(window_size, n_channels)

for i, ax in enumerate(axes.flat):
    ax.plot(sample_healthy[:500, i], label='Healthy', alpha=0.8)
    ax.plot(sample_anomaly[:500, i], label='Anomaly', alpha=0.8)
    ax.set_title(f'Bearing {i+1} Vibration')
    ax.set_xlabel('Sample')
    ax.set_ylabel('Amplitude')
    if i == 0:
        ax.legend()

plt.suptitle('Healthy vs Anomalous Vibration Patterns', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

## Step 2: Build and Train the VAE

The VAE architecture:
- **Encoder**: Compresses input to latent distribution parameters (μ, σ)
- **Latent Space**: Low-dimensional representation of sensor state
- **Decoder**: Reconstructs original sensor readings from latent code

**Key insight**: We train ONLY on healthy data. The model learns what "normal" looks like.

In [ ]:
# Define VAE architecture
input_dim = X_train.shape[1]

model = VAE(
    input_dim=input_dim,
    hidden_dims=[512, 256, 128],  # Encoder layers
    latent_dim=16,                 # Latent space dimension
    dropout=0.1,
    beta=1.0                       # β-VAE weight (higher = more regularized latent space)
)

print(f"Model architecture:")
print(f"  Input dim: {input_dim}")
print(f"  Latent dim: {model.latent_dim}")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Initialize trainer
trainer = VAETrainer(
    model=model,
    learning_rate=1e-3,
    weight_decay=1e-5,
    checkpoint_dir="checkpoints"
)

# Train on HEALTHY data only
history = trainer.train(
    X_train=X_train,
    epochs=100,
    batch_size=64,
    early_stopping_patience=15
)

In [ ]:
# Plot training history
fig = plot_training_history(history)
plt.show()

## Step 3: Anomaly Detection

Now we use the trained VAE to detect anomalies:

1. **Compute reconstruction error** for test samples
2. **Set threshold** based on training data distribution
3. **Flag anomalies** when error exceeds threshold

The key metric is **reconstruction error**: the model's inability to reproduce sensor patterns it hasn't learned.

In [ ]:
# Initialize anomaly detector with trained model
detector = AnomalyDetector(model)

# Fit threshold using training data (healthy samples)
threshold = detector.fit_threshold(
    X_train,
    method='percentile',  # Use 99th percentile of training errors
    percentile=99
)

In [ ]:
# Evaluate on test set
metrics = detector.evaluate(X_test, y_test, verbose=True)

In [ ]:
# Visualize reconstruction error distribution
test_errors = detector.get_reconstruction_errors(X_test)

fig = plot_reconstruction_error(
    errors=test_errors,
    labels=y_test,
    threshold=threshold,
    title="Reconstruction Error: Normal vs Anomaly"
)
plt.show()

In [ ]:
# Anomaly timeline view (useful for predictive maintenance)
predictions = detector.predict(X_test)

fig = plot_anomaly_timeline(
    errors=test_errors,
    predictions=predictions,
    labels=y_test,
    threshold=threshold
)
plt.show()

## Step 4: Root Cause Analysis

Once an anomaly is detected, we need to know **why**. The VAE can tell us which sensors contributed most to the reconstruction error.

This is **Feature Contribution Analysis**: examining per-feature reconstruction errors to identify the "root cause" of the anomaly.

In [ ]:
# Get anomalous samples
anomaly_mask = y_test == 1
X_anomalies = X_test[anomaly_mask]

print(f"Analyzing {len(X_anomalies)} anomalous samples...")

# Create feature names (4 bearings × window samples)
n_channels = 4
feature_names = []
for ch in range(n_channels):
    for t in range(X_train.shape[1] // n_channels):
        feature_names.append(f"Bearing_{ch+1}_t{t}")

# Analyze root causes
root_causes = detector.analyze_root_cause(
    X_anomalies[:5],  # Analyze first 5 anomalies
    feature_names=feature_names,
    top_k=20
)

In [ ]:
# Display root cause analysis for first anomaly
print("\n" + "="*60)
print("ROOT CAUSE ANALYSIS - Anomaly Sample 0")
print("="*60)

result = root_causes[0]
print(f"Total Reconstruction Error: {result['total_reconstruction_error']:.4f}")
print(f"\nTop 10 Contributing Features:")
print("-"*60)

# Group by bearing to see which bearing is the problem
bearing_contributions = {i: 0 for i in range(1, 5)}

for contrib in result['top_contributors'][:10]:
    print(f"  {contrib['rank']:2d}. {contrib['feature_name']:20s} - {contrib['contribution_pct']:5.2f}%")
    
    # Extract bearing number
    bearing_num = int(contrib['feature_name'].split('_')[1])
    bearing_contributions[bearing_num] += contrib['contribution_pct']

In [ ]:
# Aggregate analysis: Which BEARING is the root cause?
print("\n" + "="*60)
print("AGGREGATED ROOT CAUSE BY BEARING")
print("="*60)

# Compute per-bearing error across all anomalies
X_tensor = torch.FloatTensor(X_anomalies).to(detector.device)
feature_errors = detector.model.get_feature_reconstruction_error(X_tensor).cpu().numpy()

# Reshape to (samples, channels, time)
n_samples = len(X_anomalies)
n_time = X_train.shape[1] // n_channels
feature_errors_reshaped = feature_errors.reshape(n_samples, n_channels, n_time)

# Sum error per bearing
bearing_errors = feature_errors_reshaped.sum(axis=2).mean(axis=0)  # Average across samples
total_error = bearing_errors.sum()

print(f"\nBearing Error Contribution (averaged across {n_samples} anomalies):")
print("-"*40)
for i, err in enumerate(bearing_errors):
    pct = err / total_error * 100
    bar = "█" * int(pct / 2)
    print(f"  Bearing {i+1}: {pct:5.1f}% {bar}")

root_cause_bearing = np.argmax(bearing_errors) + 1
print(f"\n>>> ROOT CAUSE: Bearing {root_cause_bearing} <<<")
print("    (This matches our synthetic data where Bearing 3 was set to fail)")

In [ ]:
# Visualize bearing contributions
fig, ax = plt.subplots(figsize=(10, 6))

bearings = ['Bearing 1', 'Bearing 2', 'Bearing 3', 'Bearing 4']
percentages = bearing_errors / total_error * 100
colors = ['steelblue', 'steelblue', 'crimson', 'steelblue']  # Highlight root cause

bars = ax.bar(bearings, percentages, color=colors, edgecolor='black', linewidth=1.5)

ax.set_ylabel('Contribution to Reconstruction Error (%)')
ax.set_title('Root Cause Analysis: Error Contribution by Bearing\n(Bearing 3 is the simulated fault)', fontsize=14)
ax.set_ylim(0, max(percentages) * 1.2)

for bar, pct in zip(bars, percentages):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{pct:.1f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## Step 5: Latent Space Visualization

The VAE compresses sensor data into a low-dimensional **latent space**. Visualizing this space shows:

- **Normal samples** cluster together (the model learned their patterns)
- **Anomalies** appear as outliers (the model doesn't recognize them)

This enables **predictive maintenance**: watch points drift toward the anomaly region *before* failure.

In [ ]:
# Get latent representations
X_all = np.vstack([X_train[:500], X_test])  # Sample of training + all test
y_all = np.concatenate([np.zeros(500), y_test])  # Labels

latent_vectors = detector.get_latent_representation(X_all)
print(f"Latent vectors shape: {latent_vectors.shape}")

In [ ]:
# Visualize latent space using UMAP/t-SNE
visualizer = LatentSpaceVisualizer(method='tsne')  # Use 'umap' if installed

fig = visualizer.plot_latent_space(
    latent_vectors,
    labels=y_all,
    title="VAE Latent Space: Normal vs Anomaly Clusters",
    figsize=(12, 10)
)
plt.show()

In [ ]:
# Show latent dimensions directly (first 2 dimensions)
fig, ax = plt.subplots(figsize=(10, 8))

train_latent = detector.get_latent_representation(X_train[:500])
test_normal_latent = detector.get_latent_representation(X_test[y_test == 0])
test_anomaly_latent = detector.get_latent_representation(X_test[y_test == 1])

ax.scatter(train_latent[:, 0], train_latent[:, 1], 
           c='gray', alpha=0.3, s=20, label='Training (healthy)')
ax.scatter(test_normal_latent[:, 0], test_normal_latent[:, 1],
           c='steelblue', alpha=0.6, s=30, label='Test (healthy)')
ax.scatter(test_anomaly_latent[:, 0], test_anomaly_latent[:, 1],
           c='crimson', alpha=0.8, s=50, marker='x', linewidth=2, label='Test (anomaly)')

ax.set_xlabel('Latent Dimension 1')
ax.set_ylabel('Latent Dimension 2')
ax.set_title('Raw Latent Space (First 2 Dimensions)')
ax.legend()
plt.tight_layout()
plt.show()

## Step 6: Drift Detection (Predictive Maintenance)

In real factory settings, equipment doesn't fail instantly—it **degrades over time**.

The VAE can detect this drift by monitoring:
1. Reconstruction error trending upward
2. Latent positions moving away from the "healthy" cluster

This enables **predicting** failures days before they happen.

In [ ]:
# Simulate monitoring over time
# (In reality, this would be streaming sensor data)

# Create a timeline showing degradation
n_time_points = 100
timeline_data = []
timeline_labels = []

# Mix healthy -> degrading -> failing
for i in range(n_time_points):
    if i < 70:  # Healthy period
        idx = np.random.randint(0, len(X_train))
        timeline_data.append(X_train[idx])
        timeline_labels.append(0)
    else:  # Degradation period
        # Mix with anomalies (simulating gradual failure)
        fault_idx = np.random.randint(0, sum(y_test == 1))
        timeline_data.append(X_test[y_test == 1][fault_idx])
        timeline_labels.append(1)

timeline_data = np.array(timeline_data)
timeline_labels = np.array(timeline_labels)

# Compute errors over time
timeline_errors = detector.get_reconstruction_errors(timeline_data)

In [ ]:
# Visualize degradation timeline
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Reconstruction error over time
ax1 = axes[0]
ax1.plot(timeline_errors, linewidth=2, color='steelblue')
ax1.axhline(y=threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold')
ax1.axvline(x=70, color='orange', linestyle=':', linewidth=2, label='Degradation starts')
ax1.fill_between(range(len(timeline_errors)), 0, timeline_errors.max(),
                 where=timeline_errors > threshold, alpha=0.3, color='red')
ax1.set_ylabel('Reconstruction Error')
ax1.set_title('Predictive Maintenance: Error Trending Over Time', fontsize=14)
ax1.legend(loc='upper left')

# Ground truth
ax2 = axes[1]
ax2.fill_between(range(len(timeline_labels)), 0, timeline_labels, 
                 alpha=0.7, color='crimson', label='True fault state')
ax2.axvline(x=70, color='orange', linestyle=':', linewidth=2)
ax2.set_xlabel('Time (measurement index)')
ax2.set_ylabel('Fault State')
ax2.set_ylim(-0.1, 1.1)
ax2.set_yticks([0, 1])
ax2.set_yticklabels(['Normal', 'Fault'])

plt.tight_layout()
plt.show()

# Calculate how early we detected
first_detection = np.where(timeline_errors > threshold)[0]
if len(first_detection) > 0:
    first_detection = first_detection[0]
    lead_time = first_detection - 70  # Negative means we detected BEFORE fault started
    print(f"\n>>> PREDICTION RESULT <<<")
    print(f"    Fault started at: measurement 70")
    print(f"    First detection at: measurement {first_detection}")
    if lead_time < 0:
        print(f"    Early warning: {-lead_time} measurements BEFORE fault")
    else:
        print(f"    Detection delay: {lead_time} measurements after fault")

## Summary

This experiment demonstrated:

1. **Data Preparation**: Windowing time-series sensor data for VAE input
2. **Unsupervised Training**: The VAE learned "normal" patterns without labeled anomalies
3. **Anomaly Detection**: High reconstruction error flags anomalies
4. **Root Cause Analysis**: Per-feature error identifies which sensor/bearing is failing
5. **Latent Space**: Anomalies form distinct clusters away from normal data
6. **Predictive Maintenance**: Monitoring error trends enables early warning

### Next Steps

- Try with **real NASA Bearing data** (download from NASA Prognostics Repository)
- Experiment with **frequency-domain features** (`use_frequency_features=True`)
- Adjust **β parameter** for more disentangled latent space
- Add **Conv1D VAE** for better temporal pattern capture

In [ ]:
# Final summary metrics
print("\n" + "="*60)
print("EXPERIMENT SUMMARY")
print("="*60)
print(f"Model: VAE with {model.latent_dim}-dimensional latent space")
print(f"Training samples: {len(X_train)} (healthy only)")
print(f"Test samples: {len(X_test)} ({int(sum(y_test==0))} normal, {int(sum(y_test==1))} anomaly)")
print(f"\nPerformance:")
print(f"  AUC-ROC: {metrics['auc_roc']:.4f}")
print(f"  F1 Score: {metrics['f1_score']:.4f}")
print(f"  Precision: {metrics['precision']:.4f}")
print(f"  Recall: {metrics['recall']:.4f}")
print(f"\nRoot Cause: Bearing {root_cause_bearing} identified as source of anomaly")
print("="*60)